## 准备数据

In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [7]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [8]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([784, 128]))
        self.b1 = tf.Variable(tf.zeros([128]))
        self.W2 = tf.Variable(tf.random.normal([128, 10]))
        self.b2 = tf.Variable(tf.zeros([10]))
    
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 784])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [9]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(optimizer.learning_rate * g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [10]:
train_data, test_data = mnist_dataset()
for epoch in range(500):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())

loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 130.41861 ; accuracy 0.13095
epoch 1 : loss 128.78659 ; accuracy 0.13066667
epoch 2 : loss 127.20572 ; accuracy 0.13086666
epoch 3 : loss 125.67404 ; accuracy 0.13073333
epoch 4 : loss 124.18978 ; accuracy 0.13061666
epoch 5 : loss 122.752235 ; accuracy 0.13023333
epoch 6 : loss 121.359436 ; accuracy 0.13031666
epoch 7 : loss 120.0077 ; accuracy 0.13036667
epoch 8 : loss 118.69327 ; accuracy 0.13005
epoch 9 : loss 117.41519 ; accuracy 0.1301
epoch 10 : loss 116.17225 ; accuracy 0.1296
epoch 11 : loss 114.9625 ; accuracy 0.12931667
epoch 12 : loss 113.784515 ; accuracy 0.1287
epoch 13 : loss 112.63707 ; accuracy 0.1288
epoch 14 : loss 111.51839 ; accuracy 0.12836666
epoch 15 : loss 110.42649 ; accuracy 0.12816666
epoch 16 : loss 109.36054 ; accuracy 0.1278
epoch 17 : loss 108.32015 ; accuracy 0.12756667
epoch 18 : loss 107.30501 ; accuracy 0.12738334
epoch 19 : loss 106.31439 ; accuracy 0.1269
epoch 20 : loss 105.347786 ; accuracy 0.1268
epoch 21 : loss 104.4044 ; accurac